# Epistemic Ranker — Phase 2 (DAEDL Upgrade)

## Density-Aware EDL Integration

**Problem Diagnosed:** FIX-3 (vacuity loss targeting S=2) is a training-time patch that only works for OOD types seen during training. ERU/MI = 0.084 confirms no structural epistemic signal.

**DAEDL Solution:** Replace the point-estimate forward pass with density-scaled logits:
- **Before:** `logits → softplus → evidence → alpha`
- **After:** `logits + log(s(h)) → softplus → evidence → alpha`

where `s(h) ∈ (0, 1]` is a GDA-derived density score for CLS embedding `h`.

**Design Decisions:**
1. PCA before GDA (64 components, whiten=True)
2. QDA (class-specific covariances) not LDA
3. Prior-weighted marginal log-density
4. Sigmoid normalisation with temperature
5. Two-phase training (warm start)
6. `self.feature` side-effect pattern

**Additional Improvements:** KL annealing, cosine LR, curriculum for hard negatives, S-ordering constraint, dissonance metric, MC Dropout diagnostic.

## 1. Imports & Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set this to your actual folder in Drive
PROJECT_ROOT = '/content/drive/MyDrive/EpistemicRanker'

import os
os.chdir(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
required = ["train_pairs.parquet", "ood_slice.parquet"]
optional = ["test_4bucket.parquet"]   # rebuilt automatically if missing

for fname in required + optional:
    path = f"{PROJECT_ROOT}/data/processed/{fname}"
    tag  = "✓" if os.path.exists(path) else ("✗ MISSING (required)" if fname in required else "– will be rebuilt")
    print(f"  {tag}  {fname}")

  ✓  train_pairs.parquet
  ✓  ood_slice.parquet
  ✓  test_4bucket.parquet


In [ ]:
import os
import json
import random
import time
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from transformers import AutoModel, AutoTokenizer, set_seed, get_cosine_schedule_with_warmup

## 2. Paths & Hyperparameters

In [ ]:
# ── Paths ──
# Adjust these paths for your notebook environment
PROJECT_ROOT = Path('/content/drive/MyDrive/EpistemicRanker')
DATA_DIR     = PROJECT_ROOT / 'data'
PROCESSED    = DATA_DIR / 'processed'
REPORTS_DIR  = PROJECT_ROOT / 'reports' / 'phase2'
CKPT_DIR     = PROJECT_ROOT / 'checkpoints'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters ──
NUM_CLASSES      = 2
BATCH_SIZE       = 128       # A100 (40 GB) → 128   T4 (15 GB) → 32   V100/L4 → 64
MAX_LENGTH       = 128
EPOCHS           = 5
WARMUP_EPOCHS    = 2         # Phase A: train without density scaling
HARD_NEG_START   = 2         # Curriculum: activate hard negatives from this epoch
KL_START_EPOCH   = 3         # KL pressure delayed to epoch 3

LR_HEAD          = 5e-4
LR_LAYER11       = 1e-5
LR_LAYER10       = 5e-6
LR_LAYER9        = 1e-6
WEIGHT_DECAY     = 1e-3

LABEL_SMOOTH_EPS = 0.02
OOD_EXPOSE_EVERY = 3
OOD_LOSS_WEIGHT  = 0.5
KL_MAX_WEIGHT    = 0.4
S_REG_WEIGHT     = 0.01
S_TARGET         = 10.0
ORDERING_WEIGHT  = 0.1
ORDERING_MARGIN  = 2.0

PAIR_WEIGHTS = {
    "positive":      4.0,
    "hard_negative": 2.0,
    "easy_negative": 0.5,
}

GDA_PCA_COMPONENTS  = 64
GDA_DENSITY_TEMP    = 0.1 # prev 1.0
MC_DROPOUT_SAMPLES  = 20
MC_DROPOUT_P        = 0.10

DOUBLE_INFO = torch.finfo(torch.double)
JITTERS     = [0, DOUBLE_INFO.tiny] + [10 ** exp for exp in range(-308, 0, 1)]

## 3. Environment Setup

In [ ]:
def setup_environment():
    set_seed(42)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    print("Environment setup complete. Seed=42.")
    print(f"Python: {sys.executable}")
    print(f"Processed dir: {PROCESSED}")


if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


## 4. Data Pipeline

In [ ]:
class EpicRankerQuadDataset(Dataset):
    TYPE_ORDER = ["positive", "hard_negative", "easy_negative", "easy_negative"]

    def __init__(self, parquet_path: Path, tokenizer, max_length: int = 128):
        df              = pd.read_parquet(parquet_path)
        self.df         = df[df["pair_type"] != "ood"].copy().reset_index(drop=True)
        self.tokenizer  = tokenizer
        self.max_length = max_length
        self.grouped    = self.df.groupby("query_id")
        self.query_ids  = list(self.grouped.groups.keys())
        counts = self.df["pair_type"].value_counts().to_dict()
        print(f"[QuadDataset] {parquet_path.name}: {counts}")

    def __len__(self) -> int:
        return len(self.query_ids)

    def _get_row_for_type(self, group_df, pair_type):
        subset = group_df[group_df["pair_type"] == pair_type]
        return subset.iloc[0] if len(subset) > 0 else group_df.iloc[-1]

    def __getitem__(self, idx: int) -> dict:
        qid      = self.query_ids[idx]
        group_df = self.grouped.get_group(qid)
        seen_easy, rows = 0, []
        for t in self.TYPE_ORDER:
            if t == "easy_negative":
                easy_rows = group_df[group_df["pair_type"] == "easy_negative"]
                rows.append(easy_rows.iloc[seen_easy] if len(easy_rows) > seen_easy
                            else group_df.iloc[-1])
                seen_easy += 1
            else:
                rows.append(self._get_row_for_type(group_df, t))
        quad_df = pd.DataFrame(rows)
        enc = self.tokenizer(
            quad_df["query_text"].tolist(), quad_df["passage_text"].tolist(),
            padding="max_length", truncation=True, max_length=self.max_length,
            return_tensors="pt", return_token_type_ids=True,
        )
        return {
            "input_ids":      enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "token_type_ids": enc["token_type_ids"],
            "labels":         torch.tensor(quad_df["label"].astype(int).values,
                                           dtype=torch.long),
            "pair_types":     quad_df["pair_type"].tolist(),
        }


class OodFlatDataset(Dataset):
    def __init__(self, parquet_path: Path, tokenizer, max_length: int = 128):
        df = pd.read_parquet(parquet_path)
        if "pair_type" in df.columns:
            df = df[df["pair_type"] == "ood"].copy().reset_index(drop=True)
        self.passages   = df["passage_text"].fillna("").astype(str).tolist()
        self.queries    = (df["query_text"].fillna("").astype(str).tolist()
                           if "query_text" in df.columns else [""] * len(self.passages))
        self.pair_types = (df["pair_type"].tolist()
                           if "pair_type" in df.columns else ["ood"] * len(self.passages))
        self.tokenizer  = tokenizer
        self.max_length = max_length
        print(f"[OodFlatDataset] {parquet_path.name}: {len(self.passages)} OOD rows")

    def __len__(self) -> int:
        return len(self.passages)

    def __getitem__(self, idx: int) -> dict:
        enc = self.tokenizer(
            self.queries[idx], self.passages[idx],
            padding="max_length", truncation=True, max_length=self.max_length,
            return_tensors="pt", return_token_type_ids=True,
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc["token_type_ids"].squeeze(0),
            "pair_types":     self.pair_types[idx],
        }

## 5. Collate Functions & Input Validation

In [ ]:
def collate_quads(batch):
    return {
        "input_ids":      torch.cat([b["input_ids"]      for b in batch], dim=0),
        "attention_mask": torch.cat([b["attention_mask"]  for b in batch], dim=0),
        "token_type_ids": torch.cat([b["token_type_ids"]  for b in batch], dim=0),
        "labels":         torch.cat([b["labels"]          for b in batch], dim=0),
        "pair_types":     [pt for b in batch for pt in b["pair_types"]],
    }


def collate_flat(batch):
    return {
        "input_ids":      torch.stack([b["input_ids"]      for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"]  for b in batch]),
        "token_type_ids": torch.stack([b["token_type_ids"]  for b in batch]),
        "pair_types":     [b["pair_types"] for b in batch],
    }


def ensure_phase2_inputs():
    train_path = PROCESSED / "train_pairs.parquet"
    test_path  = PROCESSED / "test_4bucket.parquet"
    ood_path   = PROCESSED / "ood_slice.parquet"
    missing = [p for p in [train_path, ood_path] if not p.exists()]
    if missing:
        raise FileNotFoundError(f"Missing: {[str(p) for p in missing]}. Run mine.py first.")
    if not test_path.exists():
        print(f"Rebuilding {test_path.name}...")
        df_pairs = pd.read_parquet(train_path)
        df_ood   = pd.read_parquet(ood_path)
        for frame in (df_pairs, df_ood):
            for col in ("query_text", "passage_text"):
                if col in frame.columns:
                    frame[col] = frame[col].fillna("").astype(str)
            if "passage_id" in frame.columns:
                frame["passage_id"] = frame["passage_id"].astype(str)
            if "pair_type" in frame.columns:
                frame["pair_type"] = frame["pair_type"].astype(str)
        indist_sample = (df_pairs[df_pairs["pair_type"] != "ood"]
                         .groupby("pair_type", group_keys=False).head(500))
        ood_sample = df_ood.head(500).copy()
        ood_sample["query_id"] = range(-1, -len(ood_sample) - 1, -1)
        test_4bucket = pd.concat([indist_sample, ood_sample], ignore_index=True)
        test_4bucket.to_parquet(test_path, index=False, compression="snappy")
        print(f"  Created: {test_4bucket['pair_type'].value_counts().to_dict()}")
    return train_path, test_path, ood_path

## 6. GDA Density Module (DAEDL)

Pipeline: `CLS (768-dim) → PCA whitening (64-dim) → QDA per-class Gaussians → prior-weighted marginal log p(h) → sigmoid normalisation → s(h) ∈ (0,1]`

In [ ]:
class GDADensityModel:
    """
    Density-Aware EDL density estimator.

    s(h) ≈ 1.0  →  ID input   →  logits unchanged
    s(h) ≈ 0.0  →  OOD input  →  logits + log(~0) → large negative
                               →  softplus(large neg) ≈ 0 → alpha → 1 → S → K
    """

    def __init__(self, n_components=GDA_PCA_COMPONENTS, temperature=GDA_DENSITY_TEMP):
        self.n_components = n_components
        self.temperature  = temperature
        self.pca          = None
        self.gmm          = None
        self.class_means  = None
        self.class_covs   = None
        self.log_priors   = None
        self.train_mu     = None
        self.train_sigma  = None
        self.jitter_eps   = None
        self.device       = torch.device("cpu")
        self.fitted       = False

    @staticmethod
    def _gda_dtype(target_device: torch.device) -> torch.dtype:
        # CUDA/CPU support float64 well; MPS does not.
        return torch.float32 if target_device.type == "mps" else torch.double

    def to(self, target_device):
        """Move model to device and reconstruct GMM with updated dtypes."""
        target_device = torch.device(target_device)
        target_dtype  = self._gda_dtype(target_device)

        # Move stored tensors
        if self.class_means is not None:
            self.class_means = self.class_means.to(device=target_device, dtype=target_dtype)
        if self.class_covs is not None:
            self.class_covs = self.class_covs.to(device=target_device, dtype=target_dtype)
        if self.log_priors is not None:
            self.log_priors = self.log_priors.to(device=target_device, dtype=target_dtype)
        if self.train_mu is not None:
            self.train_mu = self.train_mu.to(device=target_device, dtype=target_dtype)
        if self.train_sigma is not None:
            self.train_sigma = self.train_sigma.to(device=target_device, dtype=target_dtype)

        # Move GPU-cached PCA if it exists
        if self.pca_components_gpu is not None:
            self.pca_components_gpu = self.pca_components_gpu.to(target_device)
        if self.pca_mean_gpu is not None:
            self.pca_mean_gpu = self.pca_mean_gpu.to(target_device)
        if self.pca_scale_gpu is not None:
            self.pca_scale_gpu = self.pca_scale_gpu.to(target_device)

        # Reconstruct GMM if components exist
        if self.class_means is not None and self.class_covs is not None:
            jitter = self.jitter_eps * torch.eye(
                self.n_components,
                dtype=self.class_covs.dtype,
                device=target_device,
            ).unsqueeze(0)
            self.gmm = torch.distributions.MultivariateNormal(
                loc=self.class_means,
                covariance_matrix=self.class_covs + jitter,
            )

        self.device = target_device
        return self

    def fit(self, embeddings: torch.Tensor, labels: torch.Tensor) -> None:
        """
        Full GDA fitting pipeline: PCA → QDA → prior weighting → normalisation.

        Args:
            embeddings: (N, 768) CLS embeddings from frozen model
            labels: (N,) binary labels (0=Irrelevant, 1=Relevant)
        """
        print(f"\n[GDA] Fitting on {len(embeddings):,} embeddings "
              f"(PCA {embeddings.shape[1]}→{self.n_components})...")

        # Step 1: PCA whitening
        emb_np   = embeddings.cpu().float().numpy()
        labels   = labels.cpu()
        self.pca = PCA(n_components=self.n_components, whiten=True, random_state=42)
        emb_pca  = self.pca.fit_transform(emb_np)
        emb_t    = torch.tensor(emb_pca, dtype=torch.double)

        explained = self.pca.explained_variance_ratio_.sum()
        print(f"  PCA variance explained: {explained:.1%}")

        # Step 2: Class-conditional means and covariances (QDA)
        num_classes  = int(labels.max().item()) + 1
        class_counts = torch.zeros(num_classes, dtype=torch.double)
        means, covs  = [], []

        for c in range(num_classes):
            mask     = labels == c
            count    = mask.sum().item()
            class_counts[c] = count
            cls_emb  = emb_t[mask]
            mu_c     = cls_emb.mean(dim=0)
            centered = cls_emb - mu_c
            cov_c    = (centered.T @ centered) / (count - 1)
            means.append(mu_c)
            covs.append(cov_c)
            print(f"  Class {c}: {count:,} samples")

        # Step 3: Class log-priors (prior weighting for marginal density)
        self.log_priors = torch.log(class_counts / class_counts.sum())
        print(f"  Log-priors: {[f'{p.item():.3f}' for p in self.log_priors]}")

        # Step 4: Fit MultivariateNormal with progressive jitter
        means_t = torch.stack(means)
        covs_t  = torch.stack(covs)
        gmm     = None

        # JITTERS defined globally in the notebook
        DOUBLE_INFO = torch.finfo(torch.double)
        JITTERS = [0, DOUBLE_INFO.tiny] + [10 ** exp for exp in range(-308, 0, 1)]

        for jitter_eps in JITTERS:
            try:
                jitter = jitter_eps * torch.eye(self.n_components,
                                                dtype=torch.double).unsqueeze(0)
                gmm    = torch.distributions.MultivariateNormal(
                    loc=means_t,
                    covariance_matrix=covs_t + jitter,
                )
                break
            except (RuntimeError, ValueError) as e:
                if "cholesky" in str(e).lower() or "covariance" in str(e).lower():
                    continue
                raise

        if gmm is None:
            raise RuntimeError("[GDA] Covariance not positive definite at max jitter.")

        self.class_means = means_t
        self.class_covs  = covs_t
        self.jitter_eps  = jitter_eps
        self.gmm         = gmm
        print(f"  QDA fit OK (jitter={jitter_eps:.2e})")

        # Step 5: Cache PCA components on GPU for fast inference
        self._cache_pca_gpu(self.device)

        # Step 6: Compute normalisation statistics from training set
        log_dens       = self._raw_log_density(emb_t.to(dtype=self.class_means.dtype))
        self.train_mu  = log_dens.mean()
        self.train_sigma = log_dens.std().clamp_min(1e-6)
        print(f"  Log-density stats: μ={self.train_mu:.2f}, σ={self.train_sigma:.2f}")

        self.fitted = True

    def _cache_pca_gpu(self, target_device):
        """
        Cache sklearn PCA components as GPU tensors for fast transforms.

        This eliminates the GPU→CPU→sklearn→GPU round-trip that was
        happening on every training step.
        """
        device = torch.device(target_device)
        dtype = self._gda_dtype(device)

        # Store PCA components as GPU tensors
        self.pca_components_gpu = torch.tensor(
            self.pca.components_,
            dtype=dtype,
            device=device
        )
        self.pca_mean_gpu = torch.tensor(
            self.pca.mean_,
            dtype=dtype,
            device=device
        )

        # If whitening was applied, cache the scale factors
        if self.pca.whiten:
            self.pca_scale_gpu = torch.tensor(
                np.sqrt(self.pca.explained_variance_),
                dtype=dtype,
                device=device
            )
        else:
            self.pca_scale_gpu = None

        print(f"  PCA cached on {device} ({dtype})")

    def _pca_transform_gpu(self, embeddings: torch.Tensor) -> torch.Tensor:
        """
        GPU PCA transform — replaces sklearn CPU transform.

        This is the key fix: transforms happen entirely on GPU without
        CPU/GPU synchronisation on every training step.
        Args:
            embeddings: (B, 768) on GPU
        Returns:
            (B, n_components) on GPU, dtype matching class_means
        """
        assert self.pca_components_gpu is not None, \
            "Call _cache_pca_gpu() first (happens in fit())"

        # Ensure input is on the same device as cached components
        if embeddings.device != self.pca_components_gpu.device:
            embeddings = embeddings.to(self.pca_components_gpu.device)

        # Center by training mean
        x = embeddings.to(self.pca_components_gpu.dtype) - self.pca_mean_gpu

        # Project onto components: (B, 768) @ (768, n_comp)^T = (B, n_comp)
        x = x @ self.pca_components_gpu.T

        # Apply whitening scale if it was used during PCA fit
        if self.pca_scale_gpu is not None:
            x = x / self.pca_scale_gpu

        # Return in the dtype expected by GMM
        return x.to(self.class_means.dtype)

    def _raw_log_density(self, emb_pca: torch.Tensor) -> torch.Tensor:
        """
        Compute prior-weighted marginal log-density.
        log p(h) = logsumexp_k [ log p(h|y=k) + log p(y=k) ]
        Args:
            emb_pca: (B, n_components) PCA-reduced embeddings
        Returns:
            (B,) log-densities
        """
        # Class-conditional log-probs: (B, 1, n_comp) broadcasts over K classes
        log_cls = self.gmm.log_prob(emb_pca.unsqueeze(1))   # (B, K)

        # Add log-priors and marginalise
        log_prior = self.log_priors.to(log_cls.device)
        weighted  = log_cls + log_prior.unsqueeze(0)        # (B, K)
        return torch.logsumexp(weighted, dim=-1)             # (B,)

    def score(self, embeddings: torch.Tensor) -> torch.Tensor:
        """
        Compute density score s(h) ∈ (0,1] for new embeddings.
        s(h) = sigmoid( (log p(h) - μ_train) / (σ_train * T_density) )
        In-distribution:  log p ≈ μ_train → z ≈ 0 → s ≈ 0.5
        Far OOD:          log p << μ_train → z << 0 → s → 0
        Args:
            embeddings: (B, 768) CLS embeddings on GPU
        Returns:
            (B,) ∈ (0,1] density scores
        """
        assert self.fitted, "Call fit() before score()"

        with torch.no_grad():
            # GPU PCA transform — no CPU sync
            emb_pca = self._pca_transform_gpu(embeddings)

            # Log-density
            log_p = self._raw_log_density(emb_pca)

            # Normalise by training set statistics and temperature
            mu    = self.train_mu.to(log_p.device)
            sigma = self.train_sigma.to(log_p.device)
            z     = (log_p - mu) / (sigma * self.temperature)

            # Sigmoid maps (-∞, ∞) → (0, 1)
            return torch.sigmoid(z).float()

    def density_histogram(self, embeddings: torch.Tensor,
                          pair_types: list,
                          title: str = "") -> dict:
        """
        Diagnostic: compute mean density score per pair_type slice.
        Args:
            embeddings: (N, 768) test embeddings
            pair_types: list of pair_type strings (len N)
            title: optional title for printing
        Returns:
            dict mapping pair_type → mean score
        """
        assert self.fitted, "Call fit() before density_histogram()"

        s = self.score(embeddings)
        results = {}

        for pt in ("positive", "hard_negative", "easy_negative", "ood"):
            idx = [i for i, p in enumerate(pair_types) if p == pt]
            if idx:
                results[pt] = round(s[idx].mean().item(), 4)

        if title:
            print(f"\n[GDA Density] {title}")
            for pt, v in results.items():
                # Tag slices that look correct
                if pt == "ood" and v < 0.3:
                    tag = " ← good"
                elif pt != "ood" and v > 0.3:
                    tag = " ← in-dist"
                else:
                    tag = ""
                print(f"  {pt.capitalize():>15}: s(h)={v:.4f}{tag}")

        return results

In [ ]:
# Quick sanity check
gda = GDADensityModel(n_components=64, temperature=1.0)
fake_emb = torch.randn(1000, 768).to(device)
fake_lab = torch.randint(0, 2, (1000,)).to(device)

gda.to(device)
gda.fit(fake_emb, fake_lab)

# This should not throw AttributeError
scores = gda.score(fake_emb[:100])
print(f"Scores shape: {scores.shape}, range: [{scores.min():.3f}, {scores.max():.3f}]")

NameError: name 'GDADensityModel' is not defined

## 7. Model Architecture

In [ ]:
class EvidentialHead(nn.Module):
    """
    DAEDL-aware Evidential head.
    Density scaling: logits + log(s(h))
    MC Dropout on features for offline epistemic variance diagnostic.
    """
    def __init__(self, input_dim=768, num_classes=2, dropout_p=MC_DROPOUT_P, use_dropout_in_training=False):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout_p)
        self.linear  = nn.Linear(input_dim, num_classes)
        self.register_buffer("temperature", torch.ones(1))
        self.use_dropout_in_training = use_dropout_in_training

    def set_temperature(self, temp):
        if temp <= 0:
            raise ValueError("temperature must be > 0")
        self.temperature.fill_(float(temp))

    def forward(self, features, temp=None, density_score=None, mc_dropout_active=False):
        t       = temp if temp is not None else self.temperature
        dropped = features if self.training and not self.use_dropout_in_training and not mc_dropout_active else self.dropout(features)
        logits  = self.linear(dropped) / t

        # if self.training and not self.use_dropout_in_training and not mc_dropout_active:
        #     dropped = features  # skip dropout during normal training
        # else:
        #     dropped = self.dropout(features)

        if density_score is not None:
            s      = density_score.unsqueeze(1).clamp(min=1e-6)
            logits = logits * s # logits + torch.log(s)

        evidence = F.softplus(logits)
        alpha    = evidence + 1.0
        return evidence, alpha


class BertWithEvidentialHead(nn.Module):
    """
    BERT Cross-Encoder with density-aware Evidential head.
    self.feature stores CLS embedding after each forward (GDA side-effect pattern).
    """
    def __init__(self, num_classes=2):
        super().__init__()
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        for p in self.bert.parameters():
            p.requires_grad = False
        self.bert.eval()
        self.head    = EvidentialHead(768, num_classes)
        self.feature = None

    def forward(self, input_ids=None, attention_mask=None,
                token_type_ids=None, temp=None, density_score=None, **kwargs):
        outputs      = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                                  token_type_ids=token_type_ids, **kwargs)
        cls          = outputs.last_hidden_state[:, 0, :]
        self.feature = cls
        return self.head(cls, temp=temp, density_score=density_score)

## 8. GDA Embedding Extraction

In [ ]:
# In cell 18, return pair_types alongside labels:
@torch.no_grad()
def extract_embeddings_for_gda(model, loader, gda_device):
    model.eval()
    all_embs, all_pair_codes = [], []
    pair_to_code = {"positive": 0, "hard_negative": 1, "easy_negative": 2}
    for batch in loader:
        indist_mask = torch.tensor([pt != "ood" for pt in batch["pair_types"]],
                                   dtype=torch.bool)
        if not indist_mask.any():
            continue
        model(input_ids=batch["input_ids"][indist_mask].to(gda_device),
              attention_mask=batch["attention_mask"][indist_mask].to(gda_device),
              token_type_ids=batch["token_type_ids"][indist_mask].to(gda_device),
              )
        all_embs.append(model.feature.cpu().float())
        codes = torch.tensor([pair_to_code[pt]
                              for pt, m in zip(batch["pair_types"], indist_mask) if m])
        all_pair_codes.append(codes)
    model.train()
    return torch.cat(all_embs), torch.cat(all_pair_codes)

# # In cell 18, return pair_types alongside labels:
# @torch.no_grad()
# def extract_embeddings_for_gda(model, loader, gda_device):
#     """
#     Extract CLS embeddings from in-distribution training data.
#     Called once after WARMUP_EPOCHS to fit GDA on stable representations.
#     """
#     model.eval()
#     all_embs, all_labs = [], []
#     for batch in loader:
#         indist_mask = torch.tensor([pt != "ood" for pt in batch["pair_types"]],
#                                    dtype=torch.bool)
#         if not indist_mask.any():
#             continue
#         model(
#             input_ids=batch["input_ids"][indist_mask].to(gda_device),
#             attention_mask=batch["attention_mask"][indist_mask].to(gda_device),
#             token_type_ids=batch["token_type_ids"][indist_mask].to(gda_device),
#         )
#         all_embs.append(model.feature.cpu().float())
#         all_labs.append(batch["labels"][indist_mask])
#     model.train()
#     return torch.cat(all_embs, dim=0), torch.cat(all_labs, dim=0)

## 9. Loss Functions

In [ ]:
def get_expected_probs(alpha):
    return alpha / alpha.sum(dim=1, keepdim=True).clamp_min(1e-12)


def get_smoothed_one_hot(labels, num_classes=2, epsilon=0.02):
    one_hot = torch.zeros(labels.size(0), num_classes, device=labels.device)
    one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
    return (1 - epsilon) * one_hot + (epsilon / num_classes)


def brier_score_loss(alpha, target_or_onehot, num_classes=2, sample_weights=None):
    p_hat = get_expected_probs(alpha)
    if target_or_onehot.dim() == 1:
        one_hot = torch.zeros(target_or_onehot.size(0), num_classes,
                              device=target_or_onehot.device)
        one_hot.scatter_(1, target_or_onehot.unsqueeze(1), 1.0)
    else:
        one_hot = target_or_onehot
    per_sample = torch.sum((p_hat - one_hot) ** 2, dim=1)
    if sample_weights is not None:
        per_sample = per_sample * sample_weights
    return per_sample.mean()


def binary_kl_penalty(alpha, labels, eps=1e-12):
    one_hot    = F.one_hot(labels, num_classes=NUM_CLASSES).float()
    wrong_mask = 1.0 - one_hot
    beta       = one_hot + wrong_mask * alpha
    S_beta     = beta.sum(dim=1, keepdim=True).clamp_min(eps)
    term1 = torch.lgamma(S_beta) - torch.lgamma(
        torch.tensor(float(NUM_CLASSES), device=alpha.device))
    term2 = -torch.sum(torch.lgamma(beta.clamp_min(eps)), dim=1, keepdim=True)
    term3 = torch.sum(
        (beta - 1.0) * (torch.digamma(beta.clamp_min(eps)) - torch.digamma(S_beta)),
        dim=1, keepdim=True)
    return (term1 + term2 + term3).mean()


def s_regularisation_loss(alpha, pair_types, s_target=S_TARGET):
    easy_mask = torch.tensor([pt == "easy_negative" for pt in pair_types],
                              dtype=torch.bool, device=alpha.device)
    if not easy_mask.any():
        return torch.tensor(0.0, device=alpha.device)
    return F.relu(alpha[easy_mask].sum(dim=1) - s_target).pow(2).mean()


def s_ordering_loss(alpha, pair_types, margin=ORDERING_MARGIN):
    """Enforce S(positive) > S(hard_negative) + margin."""
    S         = alpha.sum(dim=1)
    pos_mask  = torch.tensor([pt == "positive"      for pt in pair_types],
                              dtype=torch.bool, device=alpha.device)
    hard_mask = torch.tensor([pt == "hard_negative" for pt in pair_types],
                              dtype=torch.bool, device=alpha.device)
    if not pos_mask.any() or not hard_mask.any():
        return torch.tensor(0.0, device=alpha.device)
    return F.relu(S[hard_mask].mean() + margin - S[pos_mask].mean())


def vacuity_loss_fn(ood_alpha):
    """Auxiliary OOD training signal (FIX-3)."""
    return (ood_alpha.sum(dim=1) - NUM_CLASSES).pow(2).mean()


def get_kl_weight(epoch, total_epochs, max_weight=KL_MAX_WEIGHT, start_epoch=KL_START_EPOCH):
    """Anneal KL weight 0 → max_weight over training epochs."""
    if epoch < start_epoch:
        return 0.0
    return max_weight * (epoch - start_epoch + 1) / max(total_epochs - start_epoch + 1, 1)

## 10. Metrics & Profiling

In [ ]:
def base_accuracy(alpha, target):
    return (torch.argmax(get_expected_probs(alpha), dim=1) == target).float().mean()


def eru_mutual_information(alpha, eps=1e-12):
    S        = alpha.sum(dim=1, keepdim=True).clamp_min(eps)
    p_mean   = alpha / S
    pred_ent = -torch.sum(p_mean * torch.log(p_mean.clamp_min(eps)), dim=1)
    exp_ent  = -torch.sum(
        (alpha / S) * (torch.digamma(alpha + 1) - torch.digamma(S + 1)), dim=1)
    return pred_ent - exp_ent


def dirichlet_dissonance(alpha, eps=1e-12):
    """
    Subjective Logic dissonance: conflicting evidence across classes.
    OOD → low dissonance | HardNeg → higher dissonance
    """
    S    = alpha.sum(dim=1, keepdim=True).clamp_min(eps)
    b    = (alpha - 1.0) / S
    b    = b.clamp_min(0.0)
    K    = alpha.shape[1]
    diss = torch.zeros(alpha.shape[0], device=alpha.device)
    for i in range(K):
        for j in range(K):
            if i != j:
                denom  = (b[:, i] + b[:, j]).clamp_min(eps)
                bal_ij = 1.0 - (b[:, i] - b[:, j]).abs() / denom
                diss  += b[:, i] * b[:, j] * bal_ij
    return diss


def prediction_margin(alpha):
    """m = p̂_top1 - p̂_top2"""
    probs = get_expected_probs(alpha)
    top2  = probs.topk(2, dim=1).values
    return top2[:, 0] - top2[:, 1]


def _ece_from_probs(probs, target, n_bins=15):
    conf, preds = probs.max(dim=1)
    acc         = preds.eq(target).float()
    bins        = torch.linspace(0, 1, n_bins + 1, device=probs.device)
    ece         = torch.zeros(1, device=probs.device)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask   = (conf >= lo) & (conf <= hi if i == n_bins - 1 else conf < hi)
        if mask.any():
            ece += mask.float().mean() * (conf[mask].mean() - acc[mask].mean()).abs()
    return ece.squeeze(0)


@torch.no_grad()
def mc_dropout_epistemic_variance(model, loader, n_samples=MC_DROPOUT_SAMPLES):
    """Offline diagnostic: S-variance across MC Dropout passes."""
    model.train()
    all_vars = []
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        tids = batch["token_type_ids"].to(device)
        S_samples = []
        for _ in range(n_samples):
            _, alpha = model(input_ids=ids, attention_mask=mask, token_type_ids=tids)
            S_samples.append(alpha.sum(dim=1))
        S_stack = torch.stack(S_samples, dim=0)
        all_vars.append(S_stack.var(dim=0))
    model.eval()
    return torch.cat(all_vars).mean().item()


def profile_uncertainty(all_alphas, all_pair_types):
    """Groups S, dissonance, and margin by pair_type."""
    keys     = ("positive", "hard_negative", "easy_negative", "ood")
    profiles = {k: {"S": [], "dissonance": [], "margin": []} for k in keys}
    S_all    = all_alphas.sum(dim=1)
    diss_all = dirichlet_dissonance(all_alphas)
    marg_all = prediction_margin(all_alphas)

    for i, pt in enumerate(all_pair_types):
        if pt in profiles:
            profiles[pt]["S"].append(S_all[i].item())
            profiles[pt]["dissonance"].append(diss_all[i].item())
            profiles[pt]["margin"].append(marg_all[i].item())

    print("\n--- Phase 2 Uncertainty Profiles ---")
    print(f"  {'Slice':>15}  {'Mean S':>8}  {'Dissonance':>10}  {'Margin':>8}  {'n':>6}")
    print(f"  {'-'*15}  {'-'*8}  {'-'*10}  {'-'*8}  {'-'*6}")
    results = {}
    for pt, vals in profiles.items():
        if vals["S"]:
            n      = len(vals["S"])
            mean_s = sum(vals["S"]) / n
            mean_d = sum(vals["dissonance"]) / n
            mean_m = sum(vals["margin"]) / n
            results[pt] = {"mean_S": round(mean_s, 4), "mean_dissonance": round(mean_d, 4),
                           "mean_margin": round(mean_m, 4), "n": n}
            print(f"  {pt.capitalize():>15}  {mean_s:>8.4f}  {mean_d:>10.4f}  "
                  f"{mean_m:>8.4f}  {n:>6,}")
        else:
            results[pt] = None
            print(f"  {pt.capitalize():>15}  {'no samples':>28}")
    return results

## 11. Utility Helpers

In [ ]:
def get_ood_batch(ood_iter, ood_loader):
    try:
        return next(ood_iter), ood_iter
    except StopIteration:
        ood_iter = iter(ood_loader)
        return next(ood_iter), ood_iter


def quick_gradient_check(model):
    model.train()
    ids  = torch.randint(0, 30522, (4, MAX_LENGTH), device=device)
    mask = torch.ones_like(ids)
    tids = torch.zeros_like(ids)
    tgt  = torch.randint(0, NUM_CLASSES, (4,), device=device)
    _, alpha = model(input_ids=ids, attention_mask=mask, token_type_ids=tids)
    loss = brier_score_loss(alpha, tgt, num_classes=NUM_CLASSES)
    assert torch.isfinite(loss), "Loss NaN/Inf"
    model.zero_grad()
    loss.backward()
    grads = [p.grad for p in model.head.parameters() if p.requires_grad]
    assert all(g is not None for g in grads), "No head gradients"
    assert all(torch.isfinite(g).all() for g in grads), "NaN/Inf in head grads"
    print("Gradient check passed.")

## 12. Forward Pass Helper (Density-Aware or Standard)

In [ ]:
def forward_with_density(model, gda, batch_ids, batch_mask, batch_tids,
                          density_active):
    """
    Unified forward pass for eval loops.
    Applies GDA density scaling when density_active=True.
    """
    if density_active:
        outputs = model.bert(input_ids=batch_ids, attention_mask=batch_mask,
                              token_type_ids=batch_tids)
        cls = outputs.last_hidden_state[:, 0, :]
        model.feature = cls
        ds  = gda.score(cls)
        _, alpha = model.head(cls, density_score=ds)
    else:
        _, alpha = model(input_ids=batch_ids, attention_mask=batch_mask,
                          token_type_ids=batch_tids)
    return alpha

## 13. Training & Evaluation

In [ ]:
def train_and_evaluate():
    setup_environment()
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    train_path, test_path, ood_path = ensure_phase2_inputs()

    train_ds = EpicRankerQuadDataset(train_path, tokenizer, MAX_LENGTH)
    test_ds  = EpicRankerQuadDataset(test_path,  tokenizer, MAX_LENGTH)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, persistent_workers=True, collate_fn=collate_quads)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, collate_fn=collate_quads)

    ood_train_ds     = OodFlatDataset(ood_path,  tokenizer, MAX_LENGTH)
    ood_eval_ds      = OodFlatDataset(test_path, tokenizer, MAX_LENGTH)
    ood_train_loader = DataLoader(ood_train_ds, batch_size=BATCH_SIZE * 4,
                                   shuffle=True,  num_workers=0, collate_fn=collate_flat)
    ood_eval_loader  = DataLoader(ood_eval_ds,  batch_size=BATCH_SIZE * 4,
                                   shuffle=False, num_workers=0, collate_fn=collate_flat)
    ood_iter = iter(ood_train_loader)

    model = BertWithEvidentialHead(num_classes=NUM_CLASSES).to(device)
    for layer_idx in (9, 10, 11):
        for p in model.bert.encoder.layer[layer_idx].parameters():
            p.requires_grad = True
        model.bert.encoder.layer[layer_idx].train()
    model.head.set_temperature(1.0)
    quick_gradient_check(model)

    optimizer = optim.AdamW([
        {"params": model.head.parameters(),                   "lr": LR_HEAD},
        {"params": model.bert.encoder.layer[11].parameters(), "lr": LR_LAYER11},
        {"params": model.bert.encoder.layer[10].parameters(), "lr": LR_LAYER10},
        {"params": model.bert.encoder.layer[9].parameters(),  "lr": LR_LAYER9},
    ], weight_decay=WEIGHT_DECAY)

    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = total_steps // 10
    scheduler    = get_cosine_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps)

    gda = GDADensityModel(n_components=GDA_PCA_COMPONENTS,
                          temperature=GDA_DENSITY_TEMP)

    train_losses, val_accs, epoch_times = [], [], []

    print(f"\n--- Phase 2 Training (DAEDL) ---")
    print(f"  Epochs:     {EPOCHS} ({WARMUP_EPOCHS} warm-up + {EPOCHS-WARMUP_EPOCHS} density-aware)")
    print(f"  Curriculum: hard negatives from epoch {HARD_NEG_START}")
    print(f"  KL:         0 → {KL_MAX_WEIGHT} (annealed)")
    print(f"  GDA PCA:    768→{GDA_PCA_COMPONENTS} (QDA, whiten=True)")

    for epoch in range(1, EPOCHS + 1):

        # ── Phase A→B transition ──
        density_active = gda.fitted
        if epoch == WARMUP_EPOCHS + 1 and not gda.fitted:
            print(f"\n[Epoch {epoch}] Warm-up done. Fitting GDA...")
            train_embs, train_labs = extract_embeddings_for_gda(
                model, train_loader, device)
            gda.fit(train_embs, train_labs)
            gda.to(device)

            # for k in gda.models:
            #   gda.models[k].loc = gda.models[k].loc.to(device)
            #   gda.models[k].scale_tril = gda.models[k].scale_tril.to(device)

            # GDA diagnostic on eval set before Phase B
            all_diag_embs, all_diag_types = [], []
            model.eval()
            with torch.no_grad():
                for batch in test_loader:
                    model(input_ids=batch["input_ids"].to(device),
                          attention_mask=batch["attention_mask"].to(device),
                          token_type_ids=batch["token_type_ids"].to(device))
                    all_diag_embs.append(model.feature.cpu().float())
                    all_diag_types.extend(batch["pair_types"])
                for batch in ood_eval_loader:
                    model(input_ids=batch["input_ids"].to(device),
                          attention_mask=batch["attention_mask"].to(device),
                          token_type_ids=batch["token_type_ids"].to(device))
                    all_diag_embs.append(model.feature.cpu().float())
                    all_diag_types.extend(batch["pair_types"])
                    break
            gda.density_histogram(torch.cat(all_diag_embs), all_diag_types,
                                   title=f"Pre-Phase-B (epoch {epoch})")

            density_active = True
            model.train()
            for layer_idx in (9, 10, 11):
                model.bert.encoder.layer[layer_idx].train()

        kl_weight = get_kl_weight(epoch, EPOCHS, KL_MAX_WEIGHT)
        use_hard  = (epoch >= HARD_NEG_START)

        model.train()
        for layer_idx in (9, 10, 11):
            model.bert.encoder.layer[layer_idx].train()

        total_loss, total_count, batch_idx = 0.0, 0, 0
        epoch_start = time.time()

        for batch in train_loader:
            batch_idx  += 1
            input_ids   = batch["input_ids"].to(device)
            attn_mask   = batch["attention_mask"].to(device)
            token_tids  = batch["token_type_ids"].to(device)
            labels      = batch["labels"].to(device)
            pair_types  = batch["pair_types"]

            # Curriculum: zero weight on hard negatives during Phase A
            sample_weights = torch.tensor(
                [PAIR_WEIGHTS.get(pt, 1.0) if (pt != "hard_negative" or use_hard) else 0.0
                 for pt in pair_types],
                dtype=torch.float32, device=device,
            )

            one_hot = get_smoothed_one_hot(labels, NUM_CLASSES, LABEL_SMOOTH_EPS)

            # Forward — density scaling in Phase B
            outputs = model.bert(input_ids=input_ids, attention_mask=attn_mask,
                                  token_type_ids=token_tids)
            cls = outputs.last_hidden_state[:, 0, :]
            model.feature = cls

            density_score = gda.score(cls.detach()) if density_active else None
            _, alpha = model.head(cls, density_score=density_score)

            brier_loss    = brier_score_loss(alpha, one_hot, NUM_CLASSES, sample_weights)
            kl_loss       = binary_kl_penalty(alpha, labels)
            s_reg         = s_regularisation_loss(alpha, pair_types, S_TARGET)
            ordering_loss = s_ordering_loss(alpha, pair_types, ORDERING_MARGIN)
            loss = (brier_loss
                    + kl_weight        * kl_loss
                    + S_REG_WEIGHT     * s_reg
                    + ORDERING_WEIGHT  * ordering_loss)

            # OOD vacuity exposure
            if batch_idx % OOD_EXPOSE_EVERY == 0:
                ood_batch, ood_iter = get_ood_batch(ood_iter, ood_train_loader)
                ood_ids  = ood_batch["input_ids"].to(device)
                ood_mask = ood_batch["attention_mask"].to(device)
                ood_tids = ood_batch["token_type_ids"].to(device)
                ood_out  = model.bert(input_ids=ood_ids, attention_mask=ood_mask,
                                       token_type_ids=ood_tids)
                ood_cls  = ood_out.last_hidden_state[:, 0, :]
                ood_ds   = gda.score(ood_cls.detach()) if density_active else None
                _, ood_alpha = model.head(ood_cls, density_score=ood_ds)
                loss = loss + OOD_LOSS_WEIGHT * vacuity_loss_fn(ood_alpha)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()

            total_loss  += loss.item() * labels.size(0)
            total_count += labels.size(0)

        epoch_time = time.time() - epoch_start

        # Validation
        model.eval()
        va_list, vl_list = [], []
        with torch.no_grad():
            for batch in test_loader:
                alpha = forward_with_density(
                    model, gda,
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["token_type_ids"].to(device),
                    density_active)
                va_list.append(alpha)
                vl_list.append(batch["labels"].to(device))

        val_acc = base_accuracy(torch.cat(va_list), torch.cat(vl_list)).item()
        train_losses.append(total_loss / total_count)
        val_accs.append(val_acc)
        epoch_times.append(epoch_time)
        print(f"Epoch {epoch}/{EPOCHS} | {'Phase B' if density_active else 'Phase A'} | "
              f"KL={kl_weight:.3f} | Loss={train_losses[-1]:.4f} | "
              f"Val={val_acc:.4f} | {epoch_time:.1f}s")

    return model, gda, train_loader, test_loader, ood_eval_loader, density_active, train_losses, val_accs, epoch_times

## 14. Full Evaluation Suite

In [ ]:
def full_evaluation(model, gda, test_loader, ood_eval_loader, density_active,
                    train_losses, val_accs, epoch_times):
    print("\n--- Full Phase 2 Metric Suite ---")
    model.eval()

    all_alphas_indist, all_labels_indist, all_pair_types_indist = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            alpha = forward_with_density(
                model, gda,
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["token_type_ids"].to(device),
                density_active)
            all_alphas_indist.append(alpha)
            all_labels_indist.append(batch["labels"].to(device))
            all_pair_types_indist.extend(batch["pair_types"])

    alpha_indist  = torch.cat(all_alphas_indist)
    labels_indist = torch.cat(all_labels_indist)

    all_alphas_ood, all_pair_types_ood = [], []
    with torch.no_grad():
        for batch in ood_eval_loader:
            alpha = forward_with_density(
                model, gda,
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["token_type_ids"].to(device),
                density_active)
            all_alphas_ood.append(alpha)
            all_pair_types_ood.extend(batch["pair_types"])

    alpha_ood = torch.cat(all_alphas_ood)

    print("\nRunning MC Dropout epistemic variance diagnostic...")
    mc_var = mc_dropout_epistemic_variance(model, test_loader, MC_DROPOUT_SAMPLES)
    print(f"  MC Dropout S-variance (mean): {mc_var:.4f}")

    clean_acc    = base_accuracy(alpha_indist, labels_indist).item()
    clean_brier  = brier_score_loss(alpha_indist, labels_indist, NUM_CLASSES).item()
    clean_ece    = _ece_from_probs(get_expected_probs(alpha_indist), labels_indist).item()
    clean_eru    = eru_mutual_information(alpha_indist).mean().item()
    mean_S_clean = alpha_indist.sum(dim=1).mean().item()
    mean_S_ood   = alpha_ood.sum(dim=1).mean().item()

    print("\n1. Predictive Performance")
    print(f"   Accuracy : {clean_acc:.4f}")
    print(f"   Brier    : {clean_brier:.4f}")
    print(f"   ECE      : {clean_ece:.4f}")
    print("\n2. Reducible Uncertainty (free-energy proxies)")
    print(f"   ERU/MI (clean):    {clean_eru:.4f}")
    print(f"   MC Dropout S-var:  {mc_var:.4f}")
    print(f"   Mean S (clean):    {mean_S_clean:.2f}")
    print(f"   Mean S (OOD):      {mean_S_ood:.2f}  (n={len(all_pair_types_ood)}, target≈{NUM_CLASSES})")

    all_alphas_comb     = torch.cat([alpha_indist, alpha_ood])
    all_pair_types_comb = all_pair_types_indist + all_pair_types_ood
    uncertainty_profiles = profile_uncertainty(all_alphas_comb, all_pair_types_comb)

    pos_S  = (uncertainty_profiles.get("positive") or {}).get("mean_S", 0.0)
    hard_S = (uncertainty_profiles.get("hard_negative") or {}).get("mean_S", 0.0)
    ood_S  = (uncertainty_profiles.get("ood") or {}).get("mean_S", 0.0)

    routing_margin = pos_S - hard_S
    s_ratio        = ood_S / hard_S if hard_S > 0 else float("nan")

    print(f"\n3. Routing Metrics")
    print(f"   Routing margin (pos - hard_neg): {routing_margin:.3f}  (target > 3.0)")
    print(f"   OOD/HardNeg S-ratio:             {s_ratio:.3f}  (target < 0.40)")
    print(f"   DAEDL density active:            {density_active}")

    results = {
        "daedl_active":          density_active,
        "gda_pca_components":    GDA_PCA_COMPONENTS if density_active else None,
        "accuracy":              round(clean_acc,    4),
        "brier":                 round(clean_brier,  4),
        "ece":                   round(clean_ece,    4),
        "eru_mi":                round(clean_eru,    4),
        "mc_dropout_S_variance": round(mc_var,       4),
        "mean_S_clean":          round(mean_S_clean, 4),
        "mean_S_ood":            round(mean_S_ood,   4),
        "ood_eval_n":            len(all_pair_types_ood),
        "routing_margin":        round(routing_margin, 4),
        "s_ratio":               round(s_ratio, 4) if not np.isnan(s_ratio) else None,
        "uncertainty_profiles":  uncertainty_profiles,
        "train_losses":          train_losses,
        "val_accs":              val_accs,
        "epoch_times":           epoch_times,
    }
    out_path = REPORTS_DIR / "daedl_results.json"
    out_path.write_text(json.dumps(results, indent=2))
    print(f"\nResults saved → {out_path}")
    return results

## 15. Run

In [ ]:
model, gda, train_loader, test_loader, ood_eval_loader, density_active, train_losses, val_accs, epoch_times = train_and_evaluate()
results = full_evaluation(model, gda, test_loader, ood_eval_loader, density_active, train_losses, val_accs, epoch_times)

Environment setup complete. Seed=42.
Python: /usr/bin/python3
Processed dir: /content/drive/MyDrive/EpistemicRanker/data/processed
[QuadDataset] train_pairs.parquet: {'easy_negative': 100000, 'positive': 31112, 'hard_negative': 31105}
[QuadDataset] test_4bucket.parquet: {'positive': 500, 'hard_negative': 500, 'easy_negative': 500}
[OodFlatDataset] ood_slice.parquet: 5000 OOD rows
[OodFlatDataset] test_4bucket.parquet: 500 OOD rows


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Gradient check passed.

--- Phase 2 Training (DAEDL) ---
  Epochs:     5 (2 warm-up + 3 density-aware)
  Curriculum: hard negatives from epoch 2
  KL:         0 → 0.4 (annealed)
  GDA PCA:    768→64 (QDA, whiten=True)
Epoch 1/5 | Phase A | KL=0.000 | Loss=0.2529 | Val=0.5161 | 536.9s
Epoch 2/5 | Phase A | KL=0.000 | Loss=0.3335 | Val=0.7576 | 536.2s

[Epoch 3] Warm-up done. Fitting GDA...

[GDA] Fitting on 200,000 embeddings (PCA 768→64)...
  PCA variance explained: 86.9%
  Class 0: 31,112 samples
  Class 1: 31,105 samples
  Class 2: 137,783 samples
  Log-priors: ['-1.861', '-1.861', '-0.373']
  QDA fit OK (jitter=0.00e+00)
  PCA cached on cpu (torch.float64)
  Log-density stats: μ=-80.26, σ=27.92

[GDA Density] Pre-Phase-B (epoch 3)
         Positive: s(h)=0.0057
    Hard_negative: s(h)=0.0085
    Easy_negative: s(h)=0.7606 ← in-dist
              Ood: s(h)=0.0000 ← good
Epoch 3/5 | Phase B | KL=0.133 | Loss=1.0191 | Val=0.7924 | 537.2s
Epoch 4/5 | Phase B | KL=0.267 | Loss=1.0212 | V